# pYIN ablation: post-hoc and streaming voicing control

This notebook starts from the faithful local `librosa.pyin` reimplementation and evaluates two deliberately separate designs:

1. Post hoc: `pYIN + global volume gate` versus `pYIN + volume-integrated Praat-style global voicing controller`.
2. Streaming: the complete `prominence × volume gate × global unvoiced gate` 2×2×2 factorial.

The streaming unvoiced gate uses the fixed production threshold `Config.unv_thresh = 0.85`; it is not adaptive. All rows retain pYIN's frame geometry, CMNDF, beta/Boltzmann probability assignment, score-derived F0 range, and `mir_eval.melody` scoring. The implementation and reporting helpers live in `benchmarks/modules/pitch/PitchNotebook.py`.

**Rerun notice:** Earlier saved outputs predate the corrected 128-sample batch hop and HMM-free streaming implementation. Rerun those cells before interpreting their numbers. The appended smoother experiment has no saved results yet.

**After promotion:** The smoother comparison helper retains the former joint decoder as its historical reference, now labeled `legacy_joint_defaults` in fresh runs. Existing saved outputs labeled `production_defaults` refer to that same pre-promotion method. Production now uses confidence emissions with 50 ms gap bridging; validate its new benchmark results in `pitch.ipynb`.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "app.py").is_file()
    )
)
sys.path.insert(0, str(REPO_ROOT))
from benchmarks.modules.pitch.PitchNotebook import PitchNotebook


In [3]:
CONFIG = PitchNotebook.PyinAblationConfig(
    streaming_workers=8,
    volume_floor_ratio=0.02,
    volume_ceiling_percentile=95.0,
    unvoiced_threshold=0.85,
    praat_switch_cost=0.02,
)
ABLATION = PitchNotebook.PyinAblation(CONFIG)
ABLATION.show_configuration()


,setting,value
0,profile,balanced
1,datasets,"bach10-mf0-synth, urmp, coco"
2,tracks per instrument,2
3,max tracks/dataset,None
4,materialize Coco,True
5,bootstrap draws,10000
6,volume floor ratio,0.02
7,global ceiling percentile,95.0
8,global unvoiced threshold,0.85
9,Praat-style switch cost,0.02


,setting,value
0,profile,balanced
1,datasets,"bach10-mf0-synth, urmp, coco"
2,tracks per instrument,2
3,max tracks/dataset,None
4,materialize Coco,True
5,bootstrap draws,10000
6,volume floor ratio,0.02
7,global ceiling percentile,95.0
8,global unvoiced threshold,0.85
9,Praat-style switch cost,0.02


## Experimental contract

Post hoc, both variants use centered frame RMS and the same cutoff, `volume_floor_ratio * percentile(all_frame_RMS, volume_ceiling_percentile)`. The controller condition replaces the joint HMM's final voiced state with a separate whole-track maximum-score path: aggregate pYIN periodicity competes with `1 - unvoiced_threshold`, volume-ineligible frames are forbidden, and each voiced/unvoiced transition pays `praat_switch_cost` scaled to the frame step. This follows Praat's broad global path-finding structure while using pYIN evidence; it does not invoke Praat.

Streaming tests all eight combinations of prominent ACF candidate picking, a causal `volume_floor_ratio * running_peak_RMS` gate, and a frame-local `raw_unvoiced_prob < unvoiced_threshold` gate. The threshold is the fixed global production value 0.85. No future frames or dynamic unvoiced threshold are used. No band-pass filter, peak normalization, range padding, or Attune smoother is imported.

The default `balanced` profile uses a deterministic two-track-per-instrument draw from all three corpora: 8 Bach10 stems, 26 URMP stems, and 26 CocoChorales stems. CocoChorales selection is materialized on demand. Set `ATTUNE_PYIN_ABLATION_PROFILE=urmp` for the deterministic real-recording-only draw, `smoke` for a one-track plumbing check, or `full` for every available track.

In [4]:
ABLATION.check_contract()
TRACKS = ABLATION.show_selection()


Post-hoc voicing and 2×2×2 streaming contracts passed.
materialized 0 file(s) for test


tracks
dataset          instrument         
bach10-mf0-synth bassoon           2
                 clarinet          2
                 saxophone         2
                 violin            2
coco             bassoon           2
                 cello             2
                 clarinet          2
                 double bass       2
                 flute             2
                 horn              2
                 oboe              2
                 saxophone         2
                 trombone          2
                 trumpet           2
                 tuba              2
                 viola             2
                 violin            2
urmp             bassoon           2
                 cello             2
                 clarinet          2
                 double bass       2
                 flute             2
                 horn              2
                 oboe              2
                 saxophone         2
                 trombone          2
                 trumpet           2
                 tuba              2
                 viola             2
                 violin            2

60 unique tracks × 2 post-hoc or 8 streaming variants


## Paired experiment

Each selected track is evaluated by both post-hoc variants: the selected volume gate and the same volume evidence integrated with the Praat-style global voicing controller. Per-track detector caches and reports are isolated under `benchmarks/results/pitch/pyin_ablation/`. Set `ATTUNE_PYIN_ABLATION_FORCE=1` before starting the kernel to recompute them.

In [5]:
RESULTS = ABLATION.run()


pYIN post-hoc voicing controller:   0%|          | 0/120 [00:00<?, ?it/s]

Wrote 120 paired rows to /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/pyin_ablation/pyin_voicing_controllers_v1/balanced/floor_0p02__ceiling_p95__unvoiced_0p85__switch_0p02/rows.csv


## Mean metrics

Unweighted per-track means, matching the main pitch benchmark.

In [6]:
SUMMARY = ABLATION.show_summary(RESULTS)


,Overall Accuracy,Raw Pitch Accuracy,Raw Chroma Accuracy,Voicing Recall,Voicing False Alarm
variant,,,,,
pyin_volume_gate,0.9216,0.9602,0.9642,0.9874,0.3034
pyin_volume_praat_voicing,0.9283,0.9232,0.9266,0.9398,0.0739


## Paired component effects

`Gain` is oriented so positive always favors replacing the volume-only decision with the volume-integrated Praat-style global voicing controller. Confidence intervals use a dataset-stratified paired bootstrap over track-level differences.

In [ ]:
EFFECTS = ABLATION.show_effects(RESULTS)


In [ ]:
FIGURE = ABLATION.plot(RESULTS)


## URMP real-audio focus: post hoc

URMP is isolated from the aggregate report because it is the central real-recording target. These are paired comparisons on exactly the URMP rows selected above; synthetic Bach10 and rendered CocoChorales tracks cannot influence this table or its confidence intervals. To run only this corpus from the start, set `ATTUNE_PYIN_ABLATION_PROFILE=urmp` before creating `CONFIG`.

In [7]:
if "urmp" in set(RESULTS["dataset"]):
    URMP_POSTHOC_SUMMARY = ABLATION.show_dataset_summary(
        RESULTS, "urmp", streaming=False
    )
    URMP_POSTHOC_EFFECTS = ABLATION.show_dataset_effects(
        RESULTS, "urmp", streaming=False
    )
else:
    print("URMP is absent; use the balanced or urmp profile.")


,Overall Accuracy,Raw Pitch Accuracy,Raw Chroma Accuracy,Voicing Recall,Voicing False Alarm
variant,,,,,
pyin_volume_gate,0.8780,0.9529,0.9593,0.9847,0.3937
pyin_volume_praat_voicing,0.9059,0.8918,0.8969,0.9099,0.0580


,Component,Context,Metric,Tracks,Feature on,Feature off,Gain,CI low,CI high,Evidence
0,Praat-style global voicing,global volume gate present,Overall Accuracy,26,0.9059,0.8780,+0.0278,+0.0052,+0.0495,favours addition
1,Praat-style global voicing,global volume gate present,Raw Pitch Accuracy,26,0.8918,0.9529,-0.0611,-0.0965,-0.0327,favours reference
2,Praat-style global voicing,global volume gate present,Raw Chroma Accuracy,26,0.8969,0.9593,-0.0624,-0.0987,-0.0334,favours reference
3,Praat-style global voicing,global volume gate present,Voicing Recall,26,0.9099,0.9847,-0.0748,-0.1150,-0.0416,favours reference
4,Praat-style global voicing,global volume gate present,Voicing False Alarm,26,0.0580,0.3937,+0.3358,+0.2707,+0.4007,favours addition


## Streaming ablation on Attune's recording grid

This replay uses the same selected tracks and all eight cells of the prominence × volume gate × unvoiced gate factorial, but schedules every method exactly like the production recording pathway: 44.1 kHz audio, a 4096-sample integration window, a 128-sample hop, the production capture buffer, and center-of-integration timestamps. The full capture buffer must be available before an estimate is emitted.

Streaming selects the strongest raw pYIN candidate without constructing or running an HMM. The causal volume gate uses the running RMS peak; the explicit unvoiced gate uses raw frame-local confidence. All eight conditions share this frontend-only selection.

Tracks run concurrently, while all frames and eight variants for one track remain serial inside the same worker. Per-frame process CPU time drives the ideal dedicated-worker backlog simulation, so OS scheduling contention between benchmark workers does not become simulated application latency. Wall time remains a diagnostic of the benchmark host. Both this replay and the framewise pYIN comparison use 128-sample hops (2.90 ms), 4096-sample windows, and 44.1 kHz audio. The configuration cell currently requests eight workers and 12 seconds per track; edit it or set the corresponding environment variables before constructing `CONFIG`.

In [8]:
STREAMING_RESULTS = ABLATION.run_streaming()


Running 60 streaming track(s) × 8 variants with 8 worker(s).


pYIN streaming prominence × volume × unvoiced:   0%|          | 0/480 [00:00<?, ?it/s]

/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/melody.py:93: UserWarning: Reference melody has no voiced frames.
  warnings.warn("Reference melody has no voiced frames.")
/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/melody.py:93: UserWarning: Reference melody has no voiced frames.
  warnings.warn("Reference melody has no voiced frames.")
/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/melody.py:95: UserWarning: Estimated melody has no voiced frames.
  warnings.warn("Estimated melody has no voiced frames.")
/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/melody.py:95: UserWarning: Estimated melody has no voiced frames.
  warnings.warn("Estimated melody has no voiced frames.")


Wrote 480 streaming rows to /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/pyin_ablation/pyin_voicing_controllers_v1/balanced/floor_0p02__ceiling_p95__unvoiced_0p85__switch_0p02/streaming/attune_recording_grid_v5_global_unvoiced/max_12s/rows.csv


### Grid audit

The runner fails if any variant receives different sample-rate, capture-frame, integration-window, or hop geometry for the same track. The table below records that common grid; capture-frame size can vary by track because production derives `tau_max` from the score-relative `fmin`.

In [9]:
STREAMING_GRID = ABLATION.show_streaming_grid(STREAMING_RESULTS)


,tracks,sample_rate_hz,integration_size_samples,hop_size_samples,update_interval_ms,min_capture_frame_samples,max_capture_frame_samples
dataset,,,,,,,
bach10-mf0-synth,8,44100,4096,128,2.902,4096,4096
coco,26,44100,4096,128,2.902,4096,4096
urmp,26,44100,4096,128,2.902,4096,4096


### Streaming accuracy and timing

Accuracy uses the same `mir_eval.melody` metrics as the batch test. Timing is decomposed into algorithmic look-ahead, intrinsic per-call CPU service time, service/deadline utilization, simulated queue delay, and total output latency. If service/deadline exceeds 1, no wall-clock timestamp correction can keep this frame-restarted implementation current: work arrives faster than one dedicated worker can finish it.

In [10]:
STREAMING_SUMMARY = ABLATION.show_streaming_summary(STREAMING_RESULTS)


,Overall Accuracy,Raw Pitch Accuracy,Raw Chroma Accuracy,Voicing Recall,Voicing False Alarm,Audio / compute,Deadline miss rate,Algorithmic look-ahead (ms),Mean no-queue latency (ms),Median output latency (ms),P95 output latency (ms),Mean service time (ms),Service / deadline,Median queue delay (ms),P95 queue delay (ms)
variant,,,,,,,,,,,,,,,
pyin_rt_baseline,0.8052,0.9308,0.9369,0.9992,0.8081,2.6524,0.0011,46.4399,47.5426,47.5105,47.8011,1.1026,0.3799,0.0000,0.0000
pyin_rt_unvoiced,0.9394,0.8898,0.8945,0.9397,0.2594,2.6740,0.0007,46.4399,47.5337,47.5050,47.7596,1.0937,0.3768,0.0000,0.0000
pyin_rt_volume,0.8093,0.9308,0.9369,0.9991,0.7920,2.5872,0.0011,46.4399,47.5703,47.5410,47.8026,1.1304,0.3895,0.0000,0.0000
pyin_rt_volume_unvoiced,0.9398,0.8897,0.8945,0.9397,0.2548,2.5701,0.0017,46.4399,47.5779,47.5454,47.8349,1.1380,0.3921,0.0000,0.0000
pyin_rt_prominence,0.9090,0.9173,0.9310,0.9979,0.6115,2.2933,0.0027,46.4399,47.7152,47.6771,48.0141,1.2753,0.4394,0.0000,0.0017
pyin_rt_prominence_unvoiced,0.9293,0.8768,0.8822,0.9264,0.2730,2.2997,0.0027,46.4399,47.7117,47.6738,48.0082,1.2718,0.4382,0.0000,0.0050
pyin_rt_prominence_volume,0.9124,0.9172,0.9309,0.9978,0.5852,2.3067,0.0033,46.4399,47.7078,47.6712,47.9938,1.2679,0.4368,0.0000,0.0061
pyin_rt_prominence_volume_unvoiced,0.9310,0.8768,0.8821,0.9264,0.2543,2.3136,0.0024,46.4399,47.7040,47.6683,47.9844,1.2641,0.4355,0.0000,0.0017


In [ ]:
STREAMING_EFFECTS = ABLATION.show_streaming_effects(STREAMING_RESULTS)


## URMP real-audio focus: streaming

This report isolates the eight streaming conditions on URMP. It is the decisive table for whether the fixed 0.85 unvoiced gate suppresses real-room false alarms without sacrificing pitch accuracy or voicing recall, both alone and in combination with prominence and the causal volume gate.

In [11]:
if "urmp" in set(STREAMING_RESULTS["dataset"]):
    URMP_STREAMING_SUMMARY = ABLATION.show_dataset_summary(
        STREAMING_RESULTS, "urmp", streaming=True
    )
    URMP_STREAMING_EFFECTS = ABLATION.show_dataset_effects(
        STREAMING_RESULTS, "urmp", streaming=True
    )
else:
    print("URMP is absent; use the balanced or urmp profile.")


,Overall Accuracy,Raw Pitch Accuracy,Raw Chroma Accuracy,Voicing Recall,Voicing False Alarm,Audio / compute,Deadline miss rate,Algorithmic look-ahead (ms),Mean no-queue latency (ms),Median output latency (ms),P95 output latency (ms),Mean service time (ms),Service / deadline,Median queue delay (ms),P95 queue delay (ms)
variant,,,,,,,,,,,,,,,
pyin_rt_baseline,0.6039,0.8822,0.8897,0.9987,0.8131,2.3841,0.0014,46.4399,47.6667,47.6389,47.8995,1.2268,0.4227,0.0000,0.0000
pyin_rt_unvoiced,0.9338,0.8144,0.8211,0.9073,0.0260,2.3755,0.0010,46.4399,47.6711,47.6449,47.8863,1.2312,0.4242,0.0000,0.0000
pyin_rt_volume,0.6120,0.8821,0.8896,0.9987,0.7949,2.3093,0.0015,46.4399,47.7064,47.6805,47.9245,1.2665,0.4363,0.0000,0.0000
pyin_rt_volume_unvoiced,0.9340,0.8143,0.8210,0.9073,0.0256,2.3133,0.0015,46.4399,47.7042,47.6809,47.9156,1.2643,0.4356,0.0000,0.0000
pyin_rt_prominence,0.8467,0.8512,0.8741,0.9956,0.3263,2.0712,0.0031,46.4399,47.8520,47.8221,48.0988,1.4121,0.4865,0.0000,0.0038
pyin_rt_prominence_unvoiced,0.9157,0.7870,0.7957,0.8812,0.0248,2.0818,0.0030,46.4399,47.8448,47.8166,48.0824,1.4049,0.4840,0.0000,0.0058
pyin_rt_prominence_volume,0.8502,0.8512,0.8740,0.9955,0.3175,2.0828,0.0039,46.4399,47.8441,47.8142,48.0925,1.4042,0.4838,0.0000,0.0056
pyin_rt_prominence_volume_unvoiced,0.9159,0.7869,0.7956,0.8812,0.0243,2.0891,0.0035,46.4399,47.8399,47.8084,48.0995,1.4000,0.4823,0.0000,0.0039


,Component,Context,Metric,Tracks,Feature on,Feature off,Gain,CI low,CI high,Evidence
0,Prominence,"volume=off, unvoiced=off",Overall Accuracy,26,0.8467,0.6039,+0.2428,+0.1411,+0.3513,favours addition
1,Prominence,"volume=off, unvoiced=off",Raw Pitch Accuracy,26,0.8512,0.8822,-0.0309,-0.0556,-0.0111,favours reference
2,Prominence,"volume=off, unvoiced=off",Raw Chroma Accuracy,26,0.8741,0.8897,-0.0156,-0.0301,-0.0039,favours reference
3,Prominence,"volume=off, unvoiced=off",Voicing Recall,26,0.9956,0.9987,-0.0031,-0.0090,+0.0001,inconclusive
4,Prominence,"volume=off, unvoiced=off",Voicing False Alarm,26,0.3263,0.8131,+0.4868,+0.3460,+0.6235,favours addition
5,Prominence,"volume=off, unvoiced=on",Overall Accuracy,26,0.9157,0.9338,-0.0181,-0.0327,-0.0061,favours reference
6,Prominence,"volume=off, unvoiced=on",Raw Pitch Accuracy,26,0.7870,0.8144,-0.0274,-0.0506,-0.0097,favours reference
7,Prominence,"volume=off, unvoiced=on",Raw Chroma Accuracy,26,0.7957,0.8211,-0.0254,-0.0482,-0.0083,favours reference
8,Prominence,"volume=off, unvoiced=on",Voicing Recall,26,0.8812,0.9073,-0.0261,-0.0478,-0.0091,favours reference
9,Prominence,"volume=off, unvoiced=on",Voicing False Alarm,26,0.0248,0.0260,+0.0012,+0.0001,+0.0028,favours addition


## Accuracy-constrained threshold search

The fixed `0.85` experiment identifies the tradeoff but is too strict to be the promotion setting: it removes false alarms by turning many reference-voiced frames off. The search below moves toward more permissive thresholds, with extra resolution around `0.95–0.99`, and applies one explicit promotion rule: **maximize OA only among settings that also beat the named competitor on RPA on exactly the same tracks**. A convincing win requires the lower bound of the paired 95% bootstrap interval to exceed zero for both OA and RPA.

Post hoc, threshold and switch cost are searched jointly because the global path can erase short weak-pitch islands even after the local threshold is relaxed. Streaming remains causal and searches `ordinary/prominent × volume off/on × threshold`; every threshold reuses one cached pYIN evidence pass per candidate mode. These are tuning results. Freeze one setting before running the full suites in `pitch.ipynb`; confidence intervals used after selecting the best row from this same table are exploratory, not confirmatory.

In [13]:
UNVOICED_THRESHOLDS = (0.85, 0.9, 0.925, 0.95, 0.96, 0.97, 0.975, 0.98, 0.985, 0.99)
PRAAT_SWITCH_COSTS = (0.0, 0.005, 0.01, 0.02, 0.04)
VOLUME_FLOOR_RATIOS = (0.01, 0.02, 0.03, 0.04, 0.06)
from benchmarks.modules.pitch.PitchNotebook import PitchNotebook

REFERENCE_BENCH = PitchNotebook(PitchNotebook.NotebookConfig())


### Post-hoc search against Praat

This uses the 26 balanced URMP stems. The ordinary pYIN pitch-bin path is decoded once per track; every threshold × switch-cost × volume-floor setting reuses that path and the same raw periodicity/RMS evidence. `run_preliminary_urmp()` supplies Praat's rows for the identical seed-selected tracks. The volume grid is included because the threshold-only pilot crossed Praat's two point estimates by less than one tenth of a percentage point, far short of a clear paired win.

In [14]:
POSTHOC_THRESHOLD_RESULTS = ABLATION.run_posthoc_threshold_sweep(
    UNVOICED_THRESHOLDS,
    switch_costs=PRAAT_SWITCH_COSTS,
    volume_floor_ratios=VOLUME_FLOOR_RATIOS,
    dataset="urmp",
)
POSTHOC_THRESHOLD_SUMMARY = ABLATION.show_threshold_summary(
    POSTHOC_THRESHOLD_RESULTS, streaming=False
)


pYIN post-hoc threshold sweep:   0%|          | 0/26 [00:00<?, ?it/s]

Wrote 6500 post-hoc threshold rows to /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/pyin_ablation/pyin_voicing_controllers_v1/balanced/floor_0p02__ceiling_p95__unvoiced_0p85__switch_0p02/threshold_sweep/accuracy_frontier_v1/urmp_posthoc_rows.csv


In [6]:
URMP_POSTHOC_REFERENCE = REFERENCE_BENCH.run_preliminary_urmp(force=False)
POSTHOC_PROMOTION = ABLATION.show_promotion_test(
    POSTHOC_THRESHOLD_RESULTS,
    URMP_POSTHOC_REFERENCE,
    competitor="praat",
    candidate_columns=("volume_floor_ratio", "unvoiced_threshold", "praat_switch_cost"),
)


NameError: name 'REFERENCE_BENCH' is not defined

### Causal streaming search against RMVPE

The evidence cache makes this substantially cheaper than rerunning the original eight-cell factorial at every threshold: ordinary and prominent pYIN are each evaluated once per track, then the causal running-peak volume mask and frame-local unvoiced masks are applied vectorially. The comparison cell requests the same two-per-instrument URMP draw for RMVPE. It may be a long first run, but subsequent executions reuse the suite cache.

In [16]:
STREAMING_THRESHOLD_RESULTS = ABLATION.run_streaming_threshold_sweep(
    UNVOICED_THRESHOLDS,
    dataset="urmp",
    prominences=(False, True),
    volume_modes=(False, True),
    volume_floor_ratios=VOLUME_FLOOR_RATIOS,
)
STREAMING_THRESHOLD_SUMMARY = ABLATION.show_threshold_summary(
    STREAMING_THRESHOLD_RESULTS, streaming=True
)


pYIN causal unvoiced-threshold sweep:   0%|          | 0/26 [00:00<?, ?it/s]

/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/melody.py:93: UserWarning: Reference melody has no voiced frames.
  warnings.warn("Reference melody has no voiced frames.")
/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/melody.py:95: UserWarning: Estimated melody has no voiced frames.
  warnings.warn("Estimated melody has no voiced frames.")
/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/melody.py:93: UserWarning: Reference melody has no voiced frames.
  warnings.warn("Reference melody has no voiced frames.")
/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/melody.py:95: UserWarning: Estimated melody has no voiced frames.
  warnings.warn("Estimated melody has no voiced frames.")


Wrote 3120 streaming threshold rows to /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/pyin_ablation/pyin_voicing_controllers_v1/balanced/floor_0p02__ceiling_p95__unvoiced_0p85__switch_0p02/threshold_sweep/accuracy_frontier_v1/streaming/max_12s/urmp_rows.csv


In [ ]:
URMP_STREAMING_REFERENCE = REFERENCE_BENCH.run_streaming_comparison(
    instruments=(),
    tracks_per_instrument=2,
    max_seconds=CONFIG.streaming_seconds,
    force=False,
)
STREAMING_PROMOTION = ABLATION.show_promotion_test(
    STREAMING_THRESHOLD_RESULTS,
    URMP_STREAMING_REFERENCE,
    competitor="rmvpe",
    candidate_columns=(
        "prominence",
        "volume_gate",
        "volume_floor_ratio",
        "unvoiced_threshold",
    ),
)


Running low-latency live replay: 26 excerpt(s), attune, pyin_framewise, praat, swiftf0, spice, rmvpe
  skipped spice: model warm-up failed: Trying to load a model of incompatible/unknown type. '/var/folders/_1/wpkj3xlj2sxd981vsh_0fhd80000gn/T/tfhub_modules/0ec53e38da6913c097df6e04d12ced9865d7cbf9' contains neither 'saved_model.pb' nor 'saved_model.pbtxt'.
  [ 1/26] 02_Sonata_vn_vn/AuSep_2_vn_02_Sonata
  [ 2/26] 05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer
  [ 3/26] 08_Spring_fl_vn/AuSep_1_fl_08_Spring
  [ 4/26] 12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring
  [ 5/26] 14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz
  [ 6/26] 19_Pavane_cl_vn_vc/AuSep_1_cl_19_Pavane
  [ 7/26] 20_Pavane_tpt_vn_vc/AuSep_3_vc_20_Pavane
  [ 8/26] 21_Rejouissance_cl_tbn_tba/AuSep_1_cl_21_Rejouissance
  [ 9/26] 22_Rejouissance_sax_tbn_tba/AuSep_2_tbn_22_Rejouissance
  [10/26] 22_Rejouissance_sax_tbn_tba/AuSep_3_tba_22_Rejouissance
  [11/26] 24_Pirates_vn_vn_va_vc/AuSep_4_vc_24_Pirates
  [12/26] 25_Pirates_vn_vn_va_sax/AuSe

/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/melody.py:93: UserWarning: Reference melody has no voiced frames.
  warnings.warn("Reference melody has no voiced frames.")
/Users/sarah/Desktop/shasha/Attune/at-venv/lib/python3.12/site-packages/mir_eval/melody.py:95: UserWarning: Estimated melody has no voiced frames.
  warnings.warn("Estimated melody has no voiced frames.")


  [15/26] 34_Fugue_tpt_tpt_hn_tbn/AuSep_3_hn_34_Fugue
  [16/26] 35_Rondeau_vn_vn_va_db/AuSep_4_db_35_Rondeau
  [17/26] 36_Rondeau_vn_vn_va_vc/AuSep_3_va_36_Rondeau
  [18/26] 39_Jerusalem_vn_vn_va_sax_db/AuSep_5_db_39_Jerusalem
  [19/26] 40_Miserere_fl_fl_ob_cl_bn/AuSep_5_bn_40_Miserere
  [20/26] 41_Miserere_fl_fl_ob_sax_bn/AuSep_3_ob_41_Miserere
  [21/26] 41_Miserere_fl_fl_ob_sax_bn/AuSep_4_sax_41_Miserere
  [22/26] 41_Miserere_fl_fl_ob_sax_bn/AuSep_5_bn_41_Miserere
  [23/26] 42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso
  [24/26] 43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_2_tpt_43_Chorale
  [25/26] 43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_4_tbn_43_Chorale
  [26/26] 43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_5_tba_43_Chorale
Report: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/streaming_urmp/native_center_v4__methods_attune+pyin_framewise+praat+swiftf0+spice+rmvpe__instruments_all__per_instrument_2__seed_0__max_12s__dynamic_floor_0p02/rows.csv


,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,,,,,,,
attune,26,0.8512,0.8740,0.8502,0.9955,0.3175,3.7479
pyin_framewise,26,0.8815,0.8885,0.6030,0.9987,0.8137,3.2587
praat,26,0.8551,0.8792,0.8347,0.9972,0.3147,2.4033
swiftf0,26,0.7702,0.7756,0.8625,0.8667,0.0903,2.0256
rmvpe,26,0.8185,0.8250,0.8918,0.9538,0.1245,1.6663


,Transition VFA,Deep-rest VFA,Median Onset Acquisition (ms),Median Release Hangover (ms),P95 False-alarm Burst (ms),Median Output Latency (ms),P95 Output Latency (ms),Deadline Miss Rate
model,,,,,,,,
attune,0.8454,0.1710,0.0000,129.2105,365.3125,47.1834,47.6506,0.0005
pyin_framewise,0.9728,0.7637,0.0000,222.7778,1067.1731,49.6000,50.2832,0.0004
praat,0.8696,0.1643,0.0000,135.7895,112.0000,47.5058,47.7670,0.0006
swiftf0,0.3549,0.0036,27.2917,38.7500,90.6522,160.1941,160.4169,0.0005
spice,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rmvpe,0.5868,0.0003,3.1250,105.4545,80.3542,211.0668,218.9492,0.0050


Instrument  \
model  Track ID                                                        
attune 02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                   violin   
       05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer     trumpet   
       08_Spring_fl_vn/AuSep_1_fl_08_Spring                    flute   
       12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring                violin   
       14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                   flute   
...                                                              ...   
rmvpe  41_Miserere_fl_fl_ob_sax_bn/AuSep_5_bn_41_Miserere    bassoon   
       42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso        horn   
       43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_2_tpt_43_Ch...    trumpet   
       43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_4_tbn_43_Ch...   trombone   
       43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_5_tba_43_Ch...       tuba   

                                                                 Stream Adapter  \
model  Track ID                                                                   
attune 02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                   Matched app frame   
       05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer      Matched app frame   
       08_Spring_fl_vn/AuSep_1_fl_08_Spring                   Matched app frame   
       12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring                Matched app frame   
       14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                  Matched app frame   
...                                                                         ...   
rmvpe  41_Miserere_fl_fl_ob_sax_bn/AuSep_5_bn_41_Miserere  Rolling center frame   
       42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso   Rolling center frame   
       43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_2_tpt_43_Ch...  Rolling center frame   
       43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_4_tbn_43_Ch...  Rolling center frame   
       43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_5_tba_43_Ch...  Rolling center frame   

                                                           Frame Size (samples)  \
model  Track ID                                                                   
attune 02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                                4096   
       05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer                   4096   
       08_Spring_fl_vn/AuSep_1_fl_08_Spring                                4096   
       12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring                             4096   
       14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                               4096   
...                                                                         ...   
rmvpe  41_Miserere_fl_fl_ob_sax_bn/AuSep_5_bn_41_Miserere                  4960   
       42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso                   4960   
       43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_2_tpt_43_Ch...                  4960   
       43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_4_tbn_43_Ch...                  4960   
       43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_5_tba_43_Ch...                  4960   

                                                           Integration Size (samples)  \
model  Track ID                                                                         
attune 02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                                 4096.0000   
       05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer                    4096.0000   
       08_Spring_fl_vn/AuSep_1_fl_08_Spring                                 4096.0000   
       12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring                              4096.0000   
       14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                                4096.0000   
...                                                                               ...   
rmvpe  41_Miserere_fl_fl_ob_sax_bn/AuSep_5_bn_41_Miserere                         NaN   
       42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso                          NaN   
       43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_2_tpt_43_Ch...              

,prominence,volume_gate,volume_floor_ratio,unvoiced_threshold,Competitor,Tracks,Candidate OA,Competitor OA,OA margin,OA CI low,OA CI high,Candidate RPA,Competitor RPA,RPA margin,RPA CI low,RPA CI high,Clear OA+RPA win
0,False,True,0.030000,0.980000,rmvpe,26,0.9481,0.8918,+0.0563,+0.0359,+0.0784,0.8587,0.8185,+0.0402,+0.0004,+0.0749,True
1,False,True,0.040000,0.985000,rmvpe,26,0.9480,0.8918,+0.0562,+0.0364,+0.0780,0.8621,0.8185,+0.0437,+0.0071,+0.0771,True
2,False,True,0.030000,0.985000,rmvpe,26,0.9479,0.8918,+0.0561,+0.0364,+0.0776,0.8637,0.8185,+0.0453,+0.0090,+0.0783,True
3,False,True,0.020000,0.980000,rmvpe,26,0.9477,0.8918,+0.0559,+0.0354,+0.0780,0.8593,0.8185,+0.0408,+0.0011,+0.0753,True
4,False,True,0.020000,0.985000,rmvpe,26,0.9476,0.8918,+0.0558,+0.0361,+0.0773,0.8644,0.8185,+0.0460,+0.0099,+0.0787,True
5,False,False,nan,0.980000,rmvpe,26,0.9474,0.8918,+0.0556,+0.0350,+0.0778,0.8593,0.8185,+0.0409,+0.0011,+0.0754,True
6,False,True,0.010000,0.980000,rmvpe,26,0.9474,0.8918,+0.0556,+0.0350,+0.0778,0.8593,0.8185,+0.0409,+0.0011,+0.0754,True
7,False,False,nan,0.985000,rmvpe,26,0.9472,0.8918,+0.0554,+0.0356,+0.0772,0.8645,0.8185,+0.0460,+0.0099,+0.0788,True
8,False,True,0.010000,0.985000,rmvpe,26,0.9472,0.8918,+0.0554,+0.0356,+0.0772,0.8645,0.8185,+0.0460,+0.0099,+0.0788,True
9,False,True,0.060000,0.985000,rmvpe,26,0.9466,0.8918,+0.0548,+0.0351,+0.0767,0.8575,0.8185,+0.0390,+0.0024,+0.0730,True


### Promotion protocol

1. Discard every row where `Clear OA+RPA win` is false. If none survive, the tested gate family has not established the desired claim.
2. Among survivors, choose the largest OA margin; use RPA margin and simpler temporal structure as tie-breakers. Zero switch cost is a framewise periodicity controller, not a temporally global Praat-style path, and must be named that way in the paper.
3. Freeze the chosen real-time and post-hoc settings in code and in the experiment manifest. Do not retune them in `pitch.ipynb`.
4. Run the complete 149-stem URMP post-hoc suite and a full-instrument streaming suite. Report paired OA and RPA intervals against Praat/RMVPE, plus VFA and voicing recall. A full-suite claim is supported only if both OA and RPA interval lower bounds remain above zero.

## Interpretation

A component is supported only when its gain is practically meaningful and convincingly above zero on final-output metrics without an unacceptable false-alarm tradeoff. For streaming, each feature is contrasted in all four settings of the other two switches; if its direction changes, report the interaction rather than an unconditional benefit. The fixed unvoiced gate should not be accepted merely for lowering false alarms if it materially lowers raw pitch accuracy or voicing recall. An interval crossing zero does not prove equivalence.

The batch run writes `rows.csv`, `summary.csv`, `paired_effects.csv`, and `metrics.png` beneath the configured run directory. The streaming run writes its own `rows.csv`, `grid.csv`, `summary.csv`, paired caches, and `paired_effects.csv` beneath the versioned `streaming/` subdirectory.

## Preliminary: pitch-only smoothing against production defaults

This is a separate two-method experiment: the frozen joint baseline `PitchSmoother(mode='joint')` plus `VoicingSmoother`, versus `PitchSmoother(mode='pitch_only')`. It does not rerun the earlier parameter searches. Both use identical raw production candidates, 44,100 Hz / 4096-sample windows / 128-sample hops, pitch range, 0.1-semitone bins, and production voicing thresholds.

`PitchSmoother(mode='pitch_only')` computes the production voicing mask first, then decodes normalized pitch probabilities using only pitch states within each contiguous voiced run. It resets at silence; the production joint decoder can carry a hidden pitch through silence. Thus this preliminary test includes that gap-policy change. The mask must match production exactly. Candidate confidence no longer enters the pitch emissions after normalization. Octave correction still comes from temporal pitch continuity, without a voiced/unvoiced HMM.

Run the setup cells defining `ABLATION`, then jump here. Raw caches may be reused, but both smoothers are always freshly timed with warmup and alternating order. CPU/wall timings cover smoothing and final voicing only (not frontend, setup, or JIT compilation); these are post-hoc throughput measurements, not streaming latency. Three repeats are summarized by their median per track. Accuracy effects use the existing paired bootstrap. No production algorithm is promoted by this exploratory comparison.


In [18]:
from benchmarks.modules.pitch.PitchBenchmarker import PitchBenchmarker

SMOOTHER2_RESULTS, SMOOTHER2_SUMMARY, SMOOTHER2_EFFECTS, SMOOTHER2_TIMING = (
    PitchBenchmarker.run_smoother_ablation(ABLATION, repeats=3)
)
display(SMOOTHER2_SUMMARY)
display(SMOOTHER2_EFFECTS)
display(SMOOTHER2_TIMING)
print("Median paired CPU speedup:", SMOOTHER2_TIMING["speedup"].median())


Pitch-only smoother vs production:   0%|          | 0/60 [00:00<?, ?it/s]

,Raw Pitch Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,smoother_cpu_seconds,smoother_wall_seconds
variant,,,,,,
pitch_only,0.9490,0.9412,0.9768,0.1345,1.3057,1.3231
production_defaults,0.9523,0.9441,0.9768,0.1345,6.0330,6.1316


,Component,Context,Metric,Tracks,Feature on,Feature off,Gain,CI low,CI high,Evidence
0,Pitch-only smoothing,Production defaults; fixed identical voicing,Overall Accuracy,60,0.9412,0.9441,-0.0029,-0.0046,-0.0013,favours reference
1,Pitch-only smoothing,Production defaults; fixed identical voicing,Raw Pitch Accuracy,60,0.9490,0.9523,-0.0034,-0.0053,-0.0016,favours reference
2,Pitch-only smoothing,Production defaults; fixed identical voicing,Raw Chroma Accuracy,60,0.9558,0.9562,-0.0004,-0.0019,0.0007,inconclusive
3,Pitch-only smoothing,Production defaults; fixed identical voicing,Voicing Recall,60,0.9768,0.9768,0.0000,0.0000,0.0000,inconclusive
4,Pitch-only smoothing,Production defaults; fixed identical voicing,Voicing False Alarm,60,0.1345,0.1345,0.0000,0.0000,0.0000,inconclusive


variant                                                              pitch_only  \
dataset          track_id                                                         
bach10-mf0-synth 01_AchGottundHerr_violin.RESYN                          1.0907   
                 02_AchLiebenChristen_violin.RESYN                       1.1978   
                 04_ChristeDuBeistand_clarinet.RESYN                     0.6264   
                 05_DieNacht_clarinet.RESYN                              0.3253   
                 06_DieSonne_bassoon.RESYN                               1.1703   
                 07_HerrGott_saxphone.RESYN                              0.2810   
                 08_FuerDeinenThron_bassoon.RESYN                        1.0609   
                 10_NunBitten_saxphone.RESYN                             1.0065   
coco             test__brass_track222029__3_trombone                     0.3507   
                 test__brass_track222106__4_tuba                         0.2626   
                 test__brass_track222492__2_horn                         0.3591   
                 test__brass_track222970__1_trumpet                      0.5949   
                 test__brass_track223079__4_tuba                         0.5439   
                 test__brass_track223441__3_trombone                     0.3167   
                 test__brass_track223724__2_horn                         0.1952   
                 test__random_track234005__2_saxophone                   0.4336   
                 test__random_track234028__4_double bass                 0.5552   
                 test__random_track234031__4_double bass                 0.8536   
                 test__random_track234304__1_trumpet                     0.2088   
                 test__random_track234460__3_saxophone                   0.2481   
                 test__random_track235904__3_clarinet                    0.3817   
                 test__string_track216108__4_cello                       0.8363   
                 test__string_track216129__2_violin                      0.2346   
                 test__string_track216876__4_cello                       0.5143   
                 test__string_track216892__3_viola                       0.2657   
                 test__string_track217297__3_viola                       0.1904   
                 test__string_track217907__1_violin                      0.2752   
                 test__woodwind_track228039__4_bassoon                   0.6929   
                 test__woodwind_track228275__1_flute                     0.2807   
                 test__woodwind_track228702__3_clarinet                  0.5272   
                 test__woodwind_track228859__2_oboe                      0.4091   
                 test__woodwind_track228948__1_flute                     0.2185   
                 test__woodwind_track229162__4_bassoon                   0.4045   
                 test__woodwind_track229675__2_oboe                      0.1933   
urmp             02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                    0.5202   
                 05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer       2.1635   
                 08_Spring_fl_vn/AuSep_1_fl_08_Spring                    0.2537   
                 12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring                 3.9100   
                 14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                   4.7592   
                 19_Pavane_cl_vn_vc/AuSep_1_cl_19_Pavane                 7.1776   
                 20_Pavane_tpt_vn_vc/AuSep_3_vc_20_Pavane                3.6969   
                 21_Rejouissance_cl_tbn_tba/AuSep_1_cl_21_Rejoui...      1.5051   
                 22_Rejouissance_sax_tbn_tba/AuSep_2_tbn_22_Rejo...      1.2838   
                 22_Rejouissance_sax_tbn_tba/AuSep_3_tba_22_Rejo...      1.8475   
                 24_Pirates_vn_vn_va_vc/AuSep_4_vc_24_Pirates            0.6954   
                 25_Pirates_vn_vn_va_sax/AuSep_3_va_25_Pirates           1.3961   
                

Median paired CPU speedup: 3.806448833200358


## Pitch-only follow-up: bridge short gaps only

This tests only gap handling against current production defaults. `PitchSmoother(mode='pitch_only')` carries its pitch path through internal unvoiced gaps lasting at most **50 ms** (17 frames at 44.1 kHz / 128 hop). Gap frames have uniform pitch probabilities, so they add no acoustic pitch preference; they retain their original time steps. Longer gaps still reset tracking, and leading/trailing silence is skipped.

The final production voicing mask is unchanged: bridged frames remain unvoiced. Candidate probabilities on voiced frames, transition parameters, frontend and pitch grid are unchanged from the previous pitch-only experiment. No confidence-dependent emissions or parameter sweep are added. The 50 ms cutoff is a preliminary choice, not a tuned result; set `MAX_GAP_SECONDS` below to change it explicitly.

Run the setup that defines `ABLATION`, then this cell. It reloads the edited modules for an already-running kernel, reruns both smoothers, and saves results in a separate `pitch_only_gap_bridge_v1_...` directory. Earlier no-bridging results are preserved. Assess paired RPA/OA effects and speed before considering another change.

**Fast pilot:** The cell now selects up to 18 tracks, balanced across datasets with the existing fixed seed, and runs two spawned worker processes with one timed pass per method. Warmup uses only 32 frames. Tracks remain full-length; both methods see exactly the same subset. This smaller pilot has less statistical power, and single-pass timing is noisier. Use `MAX_TRACKS = None` and `REPEATS = 3` for final confirmation. CPU timings remain process-local; parallel wall timings include scheduling contention.

Selection and incremental `partial_rows.csv` are saved in a separate runner subdirectory; completed pairs survive interruption but are not automatically resumed. An already-running old cell keeps its old code: interrupt it and reload this updated cell to use these settings.

**Timing contract:** This runner uses the same `PitchDetectorBase.measure()` helper as the streaming benchmark: worker-local `process_time()` with numerical thread pools limited to one thread. Paired speedup uses median CPU service time, excluding process startup, queue wait, frontend/cache I/O, warmup and scoring. `perf_counter()` wall times are diagnostic only. Per-repeat CPU/wall samples and `timing_metadata.json` are saved for audit. CPU time excludes time spent descheduled but can still vary with CPU frequency and memory/cache contention; use three repeats and optionally `WORKERS = 1` for final timing confirmation. These are post-hoc smoothing timings, not end-to-end live latency.


In [4]:
import importlib

_smoother2_module = importlib.import_module("algorithms.PitchSmoother")
_smoother_ablation_module = importlib.import_module(
    "benchmarks.modules.pitch.PitchBenchmarker"
)
importlib.reload(_smoother2_module)
importlib.reload(_smoother_ablation_module)
MAX_GAP_SECONDS = 0.05
MAX_TRACKS = 18
WORKERS = 2
REPEATS = 1
GAP_RESULTS, GAP_SUMMARY, GAP_EFFECTS, GAP_TIMING = (
    _smoother_ablation_module.run_smoother_ablation(
        ABLATION,
        repeats=REPEATS,
        max_gap_seconds=MAX_GAP_SECONDS,
        workers=WORKERS,
        max_tracks=MAX_TRACKS,
    )
)
display(GAP_SUMMARY)
display(GAP_EFFECTS)
display(GAP_TIMING)
print("Median paired CPU speedup:", GAP_TIMING["speedup"].median())


materialized 0 file(s) for test
18 tracks; 2 workers; 1 timed pass(es) per method. Outputs: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/pyin_ablation/pyin_voicing_controllers_v2_128hop/balanced/floor_0p02__ceiling_p95__unvoiced_0p85__switch_0p02/pitch_only_gap_bridge_v1_0.05s/runner_v2_tracks_18_seed_0_repeats_1_workers_2


Pitch-only smoother vs production:   0%|          | 0/18 [00:00<?, ?it/s]

,Raw Pitch Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,smoother_cpu_seconds,smoother_wall_seconds
variant,,,,,,
pitch_only,0.961187,0.957017,0.986683,0.081876,1.869202,1.935090
production_defaults,0.968744,0.963480,0.986683,0.081876,8.904072,9.091559


,Component,Context,Metric,Tracks,Feature on,Feature off,Gain,CI low,CI high,Evidence
0,Pitch-only smoothing,Production defaults; fixed identical voicing,Overall Accuracy,18,0.957017,0.963480,-0.006463,-0.010687,-0.002595,favours reference
1,Pitch-only smoothing,Production defaults; fixed identical voicing,Raw Pitch Accuracy,18,0.961187,0.968744,-0.007557,-0.012589,-0.003128,favours reference
2,Pitch-only smoothing,Production defaults; fixed identical voicing,Raw Chroma Accuracy,18,0.968513,0.970774,-0.002261,-0.006145,0.000621,inconclusive
3,Pitch-only smoothing,Production defaults; fixed identical voicing,Voicing Recall,18,0.986683,0.986683,0.000000,0.000000,0.000000,inconclusive
4,Pitch-only smoothing,Production defaults; fixed identical voicing,Voicing False Alarm,18,0.081876,0.081876,0.000000,0.000000,0.000000,inconclusive


variant                                                              pitch_only  \
dataset          track_id                                                         
bach10-mf0-synth 01_AchGottundHerr_violin.RESYN                        1.080051   
                 02_AchLiebenChristen_violin.RESYN                     1.260028   
                 05_DieNacht_clarinet.RESYN                            0.363925   
                 07_HerrGott_saxphone.RESYN                            0.321930   
                 08_FuerDeinenThron_bassoon.RESYN                      0.914867   
                 10_NunBitten_saxphone.RESYN                           1.020853   
coco             test__brass_track223441__3_trombone                   0.356863   
                 test__random_track234005__2_saxophone                 0.468876   
                 test__string_track216108__4_cello                     0.743251   
                 test__string_track216129__2_violin                    0.232797   
                 test__string_track217297__3_viola                     0.188262   
                 test__woodwind_track229162__4_bassoon                 0.409458   
urmp             19_Pavane_cl_vn_vc/AuSep_1_cl_19_Pavane               7.273875   
                 29_Fugue_fl_fl_ob_cl/AuSep_3_ob_29_Fugue              4.412981   
                 36_Rondeau_vn_vn_va_vc/AuSep_3_va_36_Rondeau          3.514136   
                 39_Jerusalem_vn_vn_va_sax_db/AuSep_5_db_39_Jeru...    2.782131   
                 41_Miserere_fl_fl_ob_sax_bn/AuSep_3_ob_41_Miserere    0.291882   
                 42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso     8.009466   

variant                                                              production_defaults  \
dataset          track_id                                                                  
bach10-mf0-synth 01_AchGottundHerr_violin.RESYN                                 5.534337   
                 02_AchLiebenChristen_violin.RESYN                              4.704113   
                 05_DieNacht_clarinet.RESYN                                     1.171770   
                 07_HerrGott_saxphone.RESYN                                     0.931968   
                 08_FuerDeinenThron_bassoon.RESYN                               3.582723   
                 10_NunBitten_saxphone.RESYN                                    4.155526   
coco             test__brass_track223441__3_trombone                            1.112291   
                 test__random_track234005__2_saxophone                          1.543226   
                 test__string_track216108__4_cello                              2.575478   
                 test__string_track216129__2_violin                             0.910228   
                 test__string_track217297__3_viola                              0.707716   
                 test__woodwind_track229162__4_bassoon                          1.543098   
urmp             19_Pavane_cl_vn_vc/AuSep_1_cl_19_Pavane                       41.192493   
                 29_Fugue_fl_fl_ob_cl/AuSep_3_ob_29_Fugue                      21.075037   
                 36_Rondeau_vn_vn_va_vc/AuSep_3_va_36_Rondeau                  11.681912   
                 39_Jerusalem_vn_vn_va_sax_db/AuSep_5_db_39_Jeru...            10.563208   
                 41_Miserere_fl_fl_ob_sax_bn/AuSep_3_ob_41_Miserere             0.765116   
                 42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso             46.523051   

variant                                                               speedup  
dataset          track_id                                                      
bach10-mf0-synth 01_AchGottundHerr_violin.RESYN                      5.124144  
                 02_AchLiebenChristen_violin.RESYN                   3.733340  
                 05_DieNacht_clarinet.RESYN                          3.219812  
                 07_HerrGott_saxphone.RESYN                          2.894940  
                 0

Median paired CPU speedup: 3.7639217616779073


## Pitch-only follow-up: confidence-dependent emissions

This adds confidence-dependent pitch emissions to the 50 ms gap-bridging experiment, compared against **only current production defaults**. The frontend, transition matrix, pitch grid and final global voicing mask remain fixed. Both old pitch-only experiments retain their original behavior.

For each voiced frame, let `c = clip(1 - raw_unvoiced_prob, 0, 1)` and let `q` be the normalized candidate distribution. The pitch observation is `c * q + (1 - c) / number_of_pitch_bins`. High-confidence frames retain a strong candidate preference; weak frames permit surrounding pitch continuity to have more influence. Bridged unvoiced gaps still use exactly uniform observations and remain unvoiced in the output. The existing mask-equality assertion checks this on every track. This is one preliminary setting, without a parameter sweep.

Run setup defining `ABLATION`, then this cell. The pilot uses the same deterministic 18-track subset, two processes, one timed pass, and 32-frame warmup as the gap-only pilot. CPU timings use the same worker-local, single-threaded measurement helper; wall times are diagnostic. Results and partial track pairs go to a separate `pitch_only_confidence_v1_gap_...` directory. Compare paired RPA/OA and CPU speedup; an 18-track, single-pass result is exploratory.


In [5]:
import importlib

_smoother2_module = importlib.import_module("algorithms.PitchSmoother")
_smoother_ablation_module = importlib.import_module(
    "benchmarks.modules.pitch.PitchBenchmarker"
)
importlib.reload(_smoother2_module)
importlib.reload(_smoother_ablation_module)
CONFIDENCE_MAX_GAP_SECONDS = 0.05
CONFIDENCE_MAX_TRACKS = 18
CONFIDENCE_WORKERS = 2
CONFIDENCE_REPEATS = 1
CONFIDENCE_RESULTS, CONFIDENCE_SUMMARY, CONFIDENCE_EFFECTS, CONFIDENCE_TIMING = (
    _smoother_ablation_module.run_smoother_ablation(
        ABLATION,
        repeats=CONFIDENCE_REPEATS,
        max_gap_seconds=CONFIDENCE_MAX_GAP_SECONDS,
        workers=CONFIDENCE_WORKERS,
        max_tracks=CONFIDENCE_MAX_TRACKS,
        confidence_emissions=True,
    )
)
display(CONFIDENCE_SUMMARY)
display(CONFIDENCE_EFFECTS)
display(CONFIDENCE_TIMING)
print("Median paired CPU speedup:", CONFIDENCE_TIMING["speedup"].median())


18 tracks; 2 workers; 1 timed pass(es) per method. Outputs: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/pyin_ablation/pyin_voicing_controllers_v2_128hop/balanced/floor_0p02__ceiling_p95__unvoiced_0p85__switch_0p02/pitch_only_confidence_v1_gap_0.05s/runner_v2_tracks_18_seed_0_repeats_1_workers_2


Pitch-only smoother vs production:   0%|          | 0/18 [00:00<?, ?it/s]

,Raw Pitch Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,smoother_cpu_seconds,smoother_wall_seconds
variant,,,,,,
pitch_only,0.968875,0.963595,0.986683,0.081876,1.862754,1.888410
production_defaults,0.968744,0.963480,0.986683,0.081876,8.695116,8.860915


,Component,Context,Metric,Tracks,Feature on,Feature off,Gain,CI low,CI high,Evidence
0,Pitch-only smoothing,Production defaults; fixed identical voicing,Overall Accuracy,18,0.963595,0.963480,0.000115,0.000053,0.000181,favours addition
1,Pitch-only smoothing,Production defaults; fixed identical voicing,Raw Pitch Accuracy,18,0.968875,0.968744,0.000131,0.000060,0.000203,favours addition
2,Pitch-only smoothing,Production defaults; fixed identical voicing,Raw Chroma Accuracy,18,0.970905,0.970774,0.000131,0.000059,0.000205,favours addition
3,Pitch-only smoothing,Production defaults; fixed identical voicing,Voicing Recall,18,0.986683,0.986683,0.000000,0.000000,0.000000,inconclusive
4,Pitch-only smoothing,Production defaults; fixed identical voicing,Voicing False Alarm,18,0.081876,0.081876,0.000000,0.000000,0.000000,inconclusive


variant                                                              pitch_only  \
dataset          track_id                                                         
bach10-mf0-synth 01_AchGottundHerr_violin.RESYN                        1.096901   
                 02_AchLiebenChristen_violin.RESYN                     1.283236   
                 05_DieNacht_clarinet.RESYN                            0.385762   
                 07_HerrGott_saxphone.RESYN                            0.341332   
                 08_FuerDeinenThron_bassoon.RESYN                      0.935765   
                 10_NunBitten_saxphone.RESYN                           1.038711   
coco             test__brass_track223441__3_trombone                   0.381271   
                 test__random_track234005__2_saxophone                 0.500631   
                 test__string_track216108__4_cello                     0.764643   
                 test__string_track216129__2_violin                    0.245038   
                 test__string_track217297__3_viola                     0.196860   
                 test__woodwind_track229162__4_bassoon                 0.426212   
urmp             19_Pavane_cl_vn_vc/AuSep_1_cl_19_Pavane               7.178714   
                 29_Fugue_fl_fl_ob_cl/AuSep_3_ob_29_Fugue              4.475289   
                 36_Rondeau_vn_vn_va_vc/AuSep_3_va_36_Rondeau          2.913663   
                 39_Jerusalem_vn_vn_va_sax_db/AuSep_5_db_39_Jeru...    2.744058   
                 41_Miserere_fl_fl_ob_sax_bn/AuSep_3_ob_41_Miserere    0.316242   
                 42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso     8.305253   

variant                                                              production_defaults  \
dataset          track_id                                                                  
bach10-mf0-synth 01_AchGottundHerr_violin.RESYN                                 5.323602   
                 02_AchLiebenChristen_violin.RESYN                              4.689451   
                 05_DieNacht_clarinet.RESYN                                     1.172790   
                 07_HerrGott_saxphone.RESYN                                     0.928536   
                 08_FuerDeinenThron_bassoon.RESYN                               3.568537   
                 10_NunBitten_saxphone.RESYN                                    4.140197   
coco             test__brass_track223441__3_trombone                            1.113508   
                 test__random_track234005__2_saxophone                          1.559109   
                 test__string_track216108__4_cello                              2.583310   
                 test__string_track216129__2_violin                             0.915076   
                 test__string_track217297__3_viola                              0.701317   
                 test__woodwind_track229162__4_bassoon                          1.533299   
urmp             19_Pavane_cl_vn_vc/AuSep_1_cl_19_Pavane                       38.343329   
                 29_Fugue_fl_fl_ob_cl/AuSep_3_ob_29_Fugue                      20.797594   
                 36_Rondeau_vn_vn_va_vc/AuSep_3_va_36_Rondeau                  11.222996   
                 39_Jerusalem_vn_vn_va_sax_db/AuSep_5_db_39_Jeru...            10.304097   
                 41_Miserere_fl_fl_ob_sax_bn/AuSep_3_ob_41_Miserere             0.759990   
                 42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso             46.855346   

variant                                                               speedup  
dataset          track_id                                                      
bach10-mf0-synth 01_AchGottundHerr_violin.RESYN                      4.853311  
                 02_AchLiebenChristen_violin.RESYN                   3.654395  
                 05_DieNacht_clarinet.RESYN                          3.040191  
                 07_HerrGott_saxphone.RESYN                          2.720331  
                 0

Median paired CPU speedup: 3.694409845771661
